# PyPSA-USA MISO — 2025 (demo) — explore in PyPSA

Loaded straight from the [model-verse](https://modelverse.bayesian.energy) marketplace. This notebook downloads the published PyPSA network, draws the grid, and uses PyPSA's statistics module to chart installed capacity and dispatch.

_Open-source PyPSA community model. Run all cells (Runtime → Run all)._

## Setup

In [ ]:
%pip install -q pypsa netcdf4 matplotlib plotly

In [ ]:
# Config — the published NetCDF for this model
NC_URL = "https://pub-074fa9bfa2c944399f48bcd7c0d531c7.r2.dev/pypsa-usa-miso/pypsa-usa-miso-2025-demo.nc"
MODEL_NAME = "PyPSA-USA MISO — 2025 (demo)"

## Download the network

The file is served as gzip-compressed bytes, so we fetch with a browser user-agent and decompress if needed.

In [ ]:
import urllib.request, os, gzip, shutil

nc_path = "model.nc"
if not os.path.exists(nc_path):
    req = urllib.request.Request(NC_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open("model.bin", "wb") as f:
        shutil.copyfileobj(r, f)
    with open("model.bin", "rb") as f:
        is_gzip = f.read(2) == b"\x1f\x8b"
    if is_gzip:
        with gzip.open("model.bin", "rb") as fin, open(nc_path, "wb") as fout:
            shutil.copyfileobj(fin, fout)
    else:
        os.replace("model.bin", nc_path)
print(f"{os.path.getsize(nc_path)/1e6:.1f} MB ready")

## Load with PyPSA

In [ ]:
import pypsa

n = pypsa.Network(nc_path)
n

## What's in the network?

In [ ]:
print(f'Buses:      {len(n.buses)}')
print(f'Generators: {len(n.generators)}')
print(f'Lines:      {len(n.lines)}')
print(f'Links:      {len(n.links)}')
print(f'Snapshots:  {len(n.snapshots)}  ({n.snapshots[0]} → {n.snapshots[-1]})')
print(f'Carriers:   {sorted(n.generators.carrier.unique())}')

In [ ]:
n.buses.head()

In [ ]:
n.generators.head()

## Map of the network

The buses carry real coordinates, so PyPSA can draw the grid. [`n.plot.iplot()`](https://docs.pypsa.org/latest/user-guide/plotting/maps/) renders an interactive Plotly map — hover a bus or line, zoom and pan. (For a static image instead, use `n.plot.map()`.)

In [ ]:
n.plot.iplot()

## Capacity & dispatch — the statistics module

PyPSA's [`statistics`](https://docs.pypsa.org/latest/user-guide/plotting/charts/) accessor turns a solved network into ready-made charts. We first `sanitize()` so every component has a carrier (and colour) for the charts to group by.

In [ ]:
n.sanitize()

### Installed generation capacity by carrier

In [ ]:
n.statistics.installed_capacity.plot.bar(x="carrier");

### Dispatch over time

Supply by carrier stacked against demand — the system's energy balance across the modelled snapshots.

In [ ]:
n.statistics.energy_balance.plot.area(x="snapshot");

---
Explore more at [modelverse.bayesian.energy](https://modelverse.bayesian.energy) · powered by [Convexity](https://bayesian.energy).